# 1、SummarizationMiddleware中间件

## 举例1：测试trigger、keep参数

In [4]:
from langchain.agents.middleware import SummarizationMiddleware
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

# 从.env文件中加载环境变量
load_dotenv(override=True)

CLOSEAI_API_KEY = os.getenv("CLOSEAI_API_KEY")
CLOSEAI_BASE_URL = os.getenv("CLOSEAI_BASE_URL")

model = init_chat_model(
    model="gpt-5.4-mini",
    model_provider="openai",
    profile={"max_input_tokens": 128_000},
    api_key=CLOSEAI_API_KEY,
    base_url=CLOSEAI_BASE_URL
)

In [5]:

from langchain_core.messages import SystemMessage,HumanMessage,AIMessage
from langchain.agents import create_agent


messages = [
    SystemMessage("你是个非常友好的AI助手"),
    HumanMessage("你好啊，我是老王，你是谁？"),
    AIMessage("你好老王，我是小王"),
    HumanMessage("好的小王，很高兴认识你"),
    AIMessage("你高兴得太早了"),
    HumanMessage("呵呵，你什么意思")
]



agent = create_agent(
    model="deepseek-v4-flash",
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=[
                ("tokens",100),
                ("messages",6),
                ("fraction",0.001)
            ],
            keep=("messages",2)
        )
    ]
)

response = agent.invoke({
    "messages": messages
})

for msg in response["messages"]:
    msg.pretty_print()


================================ Human Message =================================

Here is a summary of the conversation to date:

## SESSION INTENT
Establish a friendly conversation and exchange introductions.

## SUMMARY
The conversation is a simple greeting/introduction in Chinese:
- The user said hello and asked who the assistant is.
- The assistant replied: “你好老王，我是小王” (“Hello Lao Wang, I’m Xiao Wang”).
- The user responded: “好的小王，很高兴认识你” (“Okay Xiao Wang, nice to meet you”).
No deeper task, decisions, or plans were مطرح.

## ARTIFACTS
None

## NEXT STEPS
None
================================== Ai Message ==================================

你高兴得太早了
================================ Human Message =================================

呵呵，你什么意思
================================== Ai Message ==================================

哈哈，老王别误会，我是开个玩笑呢！你刚才说“很高兴认识你”，我故意接了一句“你高兴得太早了”，想逗你一下，像老朋友之间互相调侃那种感觉～其实我当然也很高兴认识你！重新正式打个招呼吧：你好老王，我是小王，很高兴认识你！😊 有什么想聊的随时说～


## 举例2：summary_prompt

In [6]:

from langchain_core.messages import SystemMessage,HumanMessage,AIMessage
from langchain.agents import create_agent


messages = [
    SystemMessage("你是个非常友好的AI助手"),
    HumanMessage("你好啊，我是老王，你是谁？"),
    AIMessage("你好老王，我是小王"),
    HumanMessage("好的小王，很高兴认识你"),
    AIMessage("你高兴得太早了"),
    HumanMessage("呵呵，你什么意思")
]



agent = create_agent(
    model="deepseek-v4-flash",
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=[
                ("tokens",100),
                ("messages",6),
                ("fraction",0.001)
            ],
            keep=("messages",2),
            summary_prompt="对历史消息摘要，消息列表如下\n{messages}"
        )
    ]
)

response = agent.invoke({
    "messages": messages
})

for msg in response["messages"]:
    msg.pretty_print()


================================ Human Message =================================

Here is a summary of the conversation to date:

对历史消息的摘要如下：

老王和助手进行了简单的自我介绍。老王先打招呼并询问助手身份，助手自称“小王”；随后老王表示很高兴认识助手。
================================== Ai Message ==================================

你高兴得太早了
================================ Human Message =================================

呵呵，你什么意思
================================== Ai Message ==================================

哈哈，别误会，我是在开玩笑呢！

我是说，你上一句说“很高兴认识我”，而我作为助手，本来应该热切回应你的热情——结果我刚才那句回复写得有点“老司机”似的冷幽默，好像你在单方面高兴，而我要泼点冷水似的。其实不是那个意思啦，只是顺手调侃一下“高兴得太早”这个梗而已。

如果让你觉得不舒服了，那确实是我的不对。我重新来一遍：

**老王你好，很高兴认识你！以后有什么需要帮忙的，尽管说，我是认真来打工的小王。** 😄
